# DS 208 &middot; Programming for Data Science &mdash; Week 10 Lab
## Regular Expressions in Practice

Real text is messy &mdash; names, codes, phone numbers buried in sentences. This week you
clean it with string methods and extract from it with **regex**, a mini-language for
patterns.

**How long:** about 45 minutes. Nothing to install (`re` is built in).

Work top to bottom. The Stretch section at the end is optional.

---
## Part 0 &middot; Tidy text first

Before patterns, the everyday cleaners: `strip`, `lower`, `replace`, `split`. Run the
cell.

In [ ]:
raw = "  Cebu City ,  Davao , Iloilo  "

parts = [p.strip().lower() for p in raw.split(",")]
print(parts)
print("joined:", " | ".join(parts))

**Answer here** (double-click to edit):

1. `split(",")` then `strip()` on each piece. What would go wrong if you only split and
   did not strip?
   &rarr; *your answer*

2. `split` and `join` are opposites. In your own words, what does each one turn into what?
   &rarr; *your answer*

---
## Part 1 &middot; Find a pattern, not a literal

A regex describes a *shape* of text. `\d` is a digit, `+` means one-or-more; `findall`
returns every match. Always write patterns as `r"..."`. Run the cell.

In [ ]:
import re
text = "Order 42 shipped; order 7 pending; 128 items total."

print(re.findall(r"\d+", text))        # every run of digits
print(re.findall(r"\d{2,}", text))     # only runs of 2+ digits

**Answer here:**

1. `\d+` and `\d{2,}` returned different lists. Which numbers did the second one drop, and
   why?
   &rarr; *your answer*

2. Why write the pattern as `r"\d+"` (a raw string) rather than `"\d+"`? What does the `r`
   protect?
   &rarr; *your answer*

---
## Part 2 &middot; Match a real shape and capture parts

Parentheses `( )` **capture** pieces of a match so you can pull them out
separately. Run the cell.

In [ ]:
import re

text = "Contact 09171234567 or 09228889999 before 2024-08-31."
print("mobiles:", re.findall(r"09\d{9}", text))

m = re.search(r"(\d{4})-(\d{2})-(\d{2})", text)
print("year:", m.group(1), "month:", m.group(2), "day:", m.group(3))

**Answer here:**

1. `09\d{9}` matched both numbers. Break the pattern down: what does `09` match, and what
   does `\d{9}` add up to in total digits?
   &rarr; *your answer*

2. The date pattern used three capture groups. What did `m.group(2)` give you, and how did
   the parentheses decide that?
   &rarr; *your answer*

---
## Part 3 &middot; Regex on a whole column

pandas exposes regex through `.str`. `contains` filters; `extract` pulls a captured
group into a new column &mdash; vectorized, no loop. Run the cell.

In [ ]:
import pandas as pd
df = pd.DataFrame({"contact": ["09171234567", "no phone", "0922-888-9999", "landline"]})

df["is_mobile"] = df["contact"].str.contains(r"^09\d{9}$")
df["digits"]    = df["contact"].str.extract(r"(\d{11})")
print(df)

**Answer here:**

1. `^09\d{9}$` uses anchors `^` and `$`. Why did the dashed number `0922-888-9999` fail the
   `is_mobile` test even though it has the right digits?
   &rarr; *your answer*

2. `extract` returned `NaN` for the rows with no 11-digit run. Why is that a reasonable
   thing for it to do, rather than crashing?
   &rarr; *your answer*

---
## Stretch &mdash; optional

Stop here if you like; the required part is done.

### Stretch 1 &middot; Accept both formats

Write one pattern that matches a PH mobile whether it is written `09171234567` or
`+639171234567`. Test it with `re.findall` on a string containing both.

In [ ]:
# your code here

### Stretch 2 &middot; Pull out years

From `"born 1998, graduated 2020, tenure 2024"`, use a single `re.findall` to collect
every four-digit year into a list.

In [ ]:
# your code here

---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to
download.

You need a **submit token** &mdash; one covers every lab for a month. Open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token), sign in and generate it,
then add it **once** to Colab's Secrets panel (the &#128273; icon, left sidebar) as
`LATARAK_TOKEN`. After that the cell reads it automatically, with no prompt. No Secrets
panel? The cell will just ask, hiding what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "ds208", 10

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/ds208/lab/10/submit"
    )

# The LIVE notebook, including edits you have not saved yet.
nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]

# A month-long token. Store it once in Colab Secrets (key LATARAK_TOKEN) and
# this reads it with no prompt; otherwise it asks and hides what you type.
try:
    from google.colab import userdata
    token = (userdata.get("LATARAK_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 10 submission page](https://portal.latarak.com/course/ds208/lab/10/submit) and upload it.

Re-submitting replaces your previous attempt; the most recent version is the one kept.